# We will do the preprocessing and the analysis for the June 20x exp

In [2]:
import polars as pl
from pathlib import Path
from subprocess import run
from circadian_cell_painting import (
    detect,
    CosinorDetection,
    CosinorVisualize,
    OscillationResults,
)

analysis_ready_path = Path("/home/derfelt/PartchLabFiles/Circadian "
                           "Rhythms/Live Study/analysis_ready_data/"
                           "2026-06-01_20xRep_PHENOVUE.tsv")

preprocessed_data_dir = Path("/home/derfelt/PartchLabFiles/Circadian "
                             "Rhythms/Live Study/preprocessed_data")

pm_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live"
               " Study/platemaps/2026_05_26_JUNE-EXP_plate_map._.W-"
               "PERIODS.xlsx")
platemap = pl.read_excel(pm_path, engine='openpyxl').rename({"Well": "WellName"})

results_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/"
                   "Live Study/results")

pickle_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/L"
                  "ive Study/pickles")

visual_results_dir = Path("/home/derfelt/PartchLabFiles/Circadian R"
                          "hythms/Live Study/visuals")


display(platemap)

well_col = "WellName"
timepoint_col = "Timepoint"

MoleculeID,Human_Readable_Name,WellName,group_label,detected_period
str,str,str,str,f64
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F09""","""Chroma_DeepRed_Nucleo_ER_bmal1…",47.666667
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F07""","""Chroma_DeepRed_Nucleo_ER_bmal1…",47.666667
"""Chroma_DeepRed_Nucleo_ER_bmal1""","""no_treatment""","""F08""","""Chroma_DeepRed_Nucleo_ER_bmal1…",26.0
"""Chroma_DeepRed_Nucleo_ER_per2""","""no_treatment""","""J09""","""Chroma_DeepRed_Nucleo_ER_per2 …",28.6
"""Chroma_DeepRed_Nucleo_ER_per2""","""no_treatment""","""J07""","""Chroma_DeepRed_Nucleo_ER_per2 …",28.6
…,…,…,…,…
null,null,"""E23""","""nan | nan""",47.666667
null,null,"""F04""","""nan | nan""",47.666667
null,null,"""I06""","""nan | nan""",40.857143


detrending + denoise

In [3]:
window_hours = 28.6 # hours as per the DCT results
sampling_interval = 2 # hours
n_timepoints = round(window_hours / sampling_interval)

print(n_timepoints)

out_file = preprocessed_data_dir / f"{analysis_ready_path.stem}.parquet"
result = run(
    ['ccellpaint', 'preprocess',
     str(analysis_ready_path), str(out_file),
     '-w', well_col,
     '-t', timepoint_col,
     '--window', str(n_timepoints)],
    capture_output=False, text=True
)

if result.returncode != 0:
    print(f"ERROR: {result.stderr}")
else:
    print("Completed!")

14


INFO:root:4758 features -> 20 batches of 250
Batching progress: 100%|██████████| 20/20 [52:50<00:00, 158.51s/it]
INFO:root:denoise/detrend: window size set to: 15 timepoints
INFO:root:denoise/detrend: completed 2 pass rolling avg
INFO:root:denoise/detrend: subtracted raw signal from noise


Completed!


## Analysis portion

In [4]:
# proj params
in_file = preprocessed_data_dir / f"{analysis_ready_path.stem}.parquet"
results_out = results_dir / f"{analysis_ready_path.stem}._.RESULTS.parquet"
pickle_out = pickle_dir / f"{analysis_ready_path.stem}._.RESULT_OBJ.pickle"
csv_out = results_dir / f"{analysis_ready_path.stem}._.P-filtered.csv"

well_col = "WellName"
timepoint_col = "Timepoint"
sampling_interval = 2 # hours
stat_val = "median"
p_thresh = 0.05
period = 28.6

model = CosinorDetection(period=period)

result: OscillationResults = detect(
    file_path=in_file,
    platemap_path=platemap,
    out_file=results_out,
    well_column=well_col,
    feature_col="feature",
    timepoint_col=timepoint_col,
    value_col=stat_val,
    stat=stat_val,
    correction_group=["group_label"],
    detector=model
)

result.to_pickle(pickle_out)

(
    result.scan_results()
    .filter(pl.col("p_adjusted") <= p_thresh)
    .drop("fitted")
    .sink_csv(csv_out)
)

Well progress: 100%|██████████| 24/24 [04:04<00:00, 10.18s/it]


In [5]:
pickle_out = pickle_dir / f"{analysis_ready_path.stem}._.RESULT_OBJ.pickle"

results = OscillationResults.load_pickle(pickle_out)

viz_out = visual_results_dir / f"{analysis_ready_path.stem}"
print(viz_out)

visualize = CosinorVisualize(results=results, tick_interval=6)
visualize.view(viz_out)

/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/visuals/2026-06-01_20xRep_PHENOVUE


well_processing: 100%|██████████| 24/24 [05:27<00:00, 13.66s/it]
